# **Do Spotify audio features capture a meaningful underlying structure of music, and how does this structure compare to genre labels and popularity?**

*Carolina Leyenda, Zoe Mekin, Candela Muñoz, Carla Palmés, Sofía Paparo, Edoardo Rigoletti*

# **INTRODUCTION**

objectives and stuff

## **OVERALL ML STRATEGY**

This project is designed as a multi-perspective machine learning study rather than a single predictive task. The Spotify dataset is treated as a structured environment in which different modeling approaches can reveal complementary aspects of music.

The central idea is that the same dataset can support multiple legitimate machine learning questions, each offering a different interpretation of musical structure. In some cases, we aim to predict labels from features, such as genre or popularity. In other cases, we remove labels entirely and explore how songs organize themselves naturally. These approaches are not competing, but complementary views of the same data.

The project is therefore structured around four main analytical perspectives:

Text-based modeling, which evaluates whether textual information (track and playlist names) captures meaningful musical patterns
Popularity prediction, which examines whether musical features relate to success
Clustering, which investigates whether latent musical groupings emerge without labels
Dimensionality reduction and visualization, which provide a geometric representation of the feature space

Each perspective addresses a distinct research question while contributing to a broader understanding of how music can be represented.

At a high level, the project follows a two-stage workflow:

1. Shared Data Preparation

The dataset is first loaded, inspected, and cleaned to produce a consistent base dataset. This includes handling missing values, removing duplicates where necessary, and selecting relevant columns. The result is a clean dataset (df_clean) used across all analyses.

2. Task-Specific Modeling Pipelines

Each analytical block defines its own pipeline depending on the modeling objective. While all blocks start from the same cleaned dataset, the preprocessing steps and modeling choices differ based on the type of task being addressed.

- Text-Based Modeling (NLP + Neural Networks)
A textual representation is created by combining track and playlist names. The text is transformed into numerical features using TF-IDF. The dataset is then split into training and test sets, and classification models (Logistic Regression and a Multilayer Perceptron) are trained to evaluate whether textual information captures meaningful musical patterns.
- Popularity Prediction
The target variable is defined as a categorical version of track popularity (e.g., low, medium, high). Separate feature sets are constructed using audio features, textual features, and their combination. Numerical features are scaled where appropriate. The dataset is split, and multiple models (Logistic Regression, Random Forest, Gradient Boosting) are trained and compared to assess how well popularity can be predicted from different types of information.
- Clustering (Unsupervised Learning)
Only numerical audio features are used. These features are scaled to ensure comparability. Dimensionality reduction (PCA) may be applied before clustering. K-Means (and optionally GMM) is used to identify latent groupings in the data without using labels. Cluster quality is evaluated using metrics such as silhouette score, and clusters are later compared to genre and popularity.
- Dimensionality Reduction and Visualization
PCA (and optionally UMAP) is applied to project high-dimensional feature spaces into two dimensions. These projections are visualized by coloring points according to genre, popularity, or cluster assignments. This step provides an interpretable geometric view of the relationships uncovered in the previous analyses.

This design improves interpretability and analytical depth. If the project relied on a single target, conclusions would remain limited. By combining classification, prediction, clustering, and visualization, we can compare what aspects of music are easily captured by the data and what are not.

For example, if genre can be predicted accurately but popularity cannot, this suggests that musical structure is well represented in the features, while success depends on external factors beyond the dataset. Similarly, if clustering reveals patterns that do not align with genre labels, this highlights the limitations of predefined categories in representing musical reality.

In summary, the project uses a shared cleaned dataset and multiple machine learning perspectives to provide a richer and more comprehensive analysis of music.

# **1. DATA PREPARATION**

The first step is to load the dataset and understand what we're actually working with before building any models. This means checking the structure of the data, the types of variables, and what each column represents.

The dataset contains different kinds of information. Some columns are identifiers (like track or playlist IDs), which are not useful for prediction. Others are descriptive text, such as track names or artists, which may carry some contextual information. We also have playlist-level labels like genre and subgenre, the popularity score, and a set of numerical audio features (such as danceability, energy, tempo, etc.), which describe the music itself.

This distinction is important for the project. Audio features represent intrinsic properties of the songs, while text and playlist labels reflect external or contextual information. Since our goal is to compare different representations of music, separating these groups early helps us stay consistent in how we use them later.

For the supervised part of the project, we use track popularity as the target variable. It is originally a numerical value between 0 and 100, but instead of predicting the exact value, we convert it into three categories: low, medium, and high popularity. This makes the problem easier to interpret and better suited for classification models. The categories are defined using quantiles to ensure a balanced distribution of classes.

During exploration, we also look at the distribution of the main variables. For numerical features, we check ranges and general shape (for example, whether values are skewed or concentrated). For categorical variables like genre, we look at how balanced the classes are. We also inspect the distribution of popularity. These checks are important because imbalance or overlapping distributions can affect our models' performance and our interpretation of the results.

One important observation is that **the same track can appear multiple times across different playlists**. This means the dataset is playlist-based rather than track-based, so the observations are not fully independent. We decided not to remove the duplicates, because they represent different contexts in each playlist, and if we deleted them, we would be losing some playlist/context information. But we also have to acknowledge that this may introduce bias and it needs to be considered during interpretation.

## 1.1 FIRST LOOK AT THE DATA

In [ ]:
# IMPORTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# DISPLAY AND PLOT SETTINGS

pd.set_option("display.max_columns", None) # Display settings
pd.set_option("display.width", 120)
sns.set(style="whitegrid") # Plot settings

In [ ]:
# LOAD DATASET

df = pd.read_csv("spotify_songs.csv")
print(f"Shape: {df.shape}") # Rows = samples ; Columns = features ; 33k observations, 23 features

The dataset contains 32833 observations and 23 features. This size is sufficient for both supervised and unsupervised learning tasks. We should have more than enough data to find significant patterns.

In [ ]:
display(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes) # Numeric vs categorical variables

df.info()

A quick inspection of the dataset confirms that each row corresponds to a track within a playlist. The data includes a mix of identifiers, textual information, and numerical audio features, which will need to be handled differently during preprocessing.

## 1.2 DESCRIPTIVE STATISTICS

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns
categorical_cols = df.select_dtypes(exclude=np.number).columns

display(df[numerical_cols].describe().T)
display(df[categorical_cols].describe().T)

The numerical summary shows that audio features are generally well-scaled within expected ranges, although some variables (such as tempo and duration) have wider ranges. Feature scaling will be important for certain models later on.

## 1.3 CLASS DISTRIBUTION (GENRE AND SUBGENRE)

In [ ]:
if "playlist_genre" in df.columns:
    plt.figure(figsize=(10, 5))
    genre_counts = df["playlist_genre"].value_counts()
    sns.barplot(x=genre_counts.index, y=genre_counts.values)
    plt.title("Distribution of Playlist Genres")
    plt.xlabel("Genre")
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()

if "playlist_subgenre" in df.columns:
    plt.figure(figsize=(12, 6))
    subgenre_counts = df["playlist_subgenre"].value_counts().head(15)
    sns.barplot(x=subgenre_counts.index, y=subgenre_counts.values)
    plt.title("Top 15 Playlist Subgenres")
    plt.xlabel("Subgenre")
    plt.ylabel("Count")
    plt.xticks(rotation=60, ha="right")
    plt.show()

The distribution of genres and subgenres is not perfectly balanced, with some categories appearing much more frequently than others. This imbalance can affect classification performance and reinforces the need for appropriate evaluation metrics such as macro F1.

## 1.4 POPULARITY DISTRIBUTION

In [ ]:
if "track_popularity" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df["track_popularity"], bins=30, kde=True)
    plt.title("Distribution of Track Popularity")
    plt.xlabel("Track Popularity")
    plt.ylabel("Frequency")
    plt.show()

Track popularity is spread across the full range from 0 to 100, without extreme concentration in a single region. This supports transforming it into balanced categories for classification using quantiles rather than treating it as a regression problem.

## 1.5 AUDIO FEATURES DISTRIBUTION

In [ ]:
audio_features = [
    "danceability", "energy", "key", "loudness", "mode", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
    "tempo", "duration_ms"]

audio_features = [col for col in audio_features if col in df.columns]

df[audio_features].hist(figsize=(16, 12), bins=30)
plt.suptitle("Distributions of Core Audio Features", fontsize=16)
plt.tight_layout()
plt.show()

The distributions of audio features vary across variables, with some showing skewness or concentration around specific values. Different features may carry different levels of information and may require scaling or transformation.

## 1.6 CORRELATION HEATMAP

In [ ]:
plt.figure(figsize=(12, 8))
corr_matrix = df[audio_features + (["track_popularity"] if "track_popularity" in df.columns else [])].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Audio Features")
plt.show()

The correlation analysis shows generally weak relationships between individual audio features and track popularity. This means that popularity may not be strongly explained by audio features alone, which aligns with the idea that external factors also play a role.

Some artists appear much more frequently than others; there may be potential bias in the dataset. Moreover, repeated tracks confirm that the same song can appear in multiple contexts, as mentioned earlier.

## 1.7 ARTIST AND TRACK FREQUENCY

In [ ]:
# artists

if "track_artist" in df.columns:
    top_artists = df["track_artist"].value_counts().head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_artists.values, y=top_artists.index)
    plt.title("Top 10 Most Frequent Artists in the Dataset")
    plt.xlabel("Number of Songs")
    plt.ylabel("Artist")
    plt.show()

# REPEATED TRACKS

if "track_id" in df.columns:
    repeated_track_counts = df["track_id"].value_counts() # If we count by track name instead of ID, we also get different counts because different songs have the same names
    repeated_tracks = repeated_track_counts[repeated_track_counts > 1]
    print(f"\nNumber of tracks appearing more than once: {repeated_tracks.shape[0]}")
    display(repeated_tracks.head(10))

## 1.8 HANDLING MISSING AND DUPLICATE VALUES

We begin the data preparation process by handling missing values. Since the dataset contains only a very small number of missing observations, we choose to remove these rows entirely. Given the size of the dataset, this has a negligible impact on the analysis while ensuring data consistency and avoiding the need for imputation.

In [ ]:
# Count missing values per column
missing_counts = df.isnull().sum()

# Show only columns with missing values
missing_counts = missing_counts[missing_counts > 0]

print("Missing values per column:")
print(missing_counts)

# Total missing values
print("\nTotal missing values:", df.isnull().sum().sum())

In [ ]:
# Handle missing values by dropping rows with any missing entries
df_clean = df.dropna()

print("Original shape:", df.shape)
print("Shape after removing missing values:", df_clean.shape)

# Verify no missing values remain
print("Remaining missing values:", df_clean.isnull().sum().sum())

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

The next step would be to remove full duplicates but since we already found that there are none, and we don't intend to remove the songs that are the same but are in different playlists, we will skip this part.

After cleaning the dataset, we select only the relevant columns required for the analysis. This includes textual information, target variables, and core audio features. Removing unnecessary columns simplifies the dataset and ensures that subsequent modeling steps remain focused and efficient

## 1.9 COLUMN DEFINITION

In [ ]:
# Define relevant columns for the project
columns_to_keep = [
    "track_name",
    "playlist_name",
    "playlist_genre",
    "playlist_subgenre",
    "track_popularity",
    "track_artist",
    "track_album_name",
    "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "duration_ms"
]

# Keep only selected columns
df_clean = df_clean[columns_to_keep]

print("Final dataset shape:", df_clean.shape)
display(df_clean.head())

---

# **2. TASK-SPECIFIC PIPELINES**

## **A. TEXT-BASED MODELING**

## A.1 PROBLEM DEFINITION AND TARGET VARIABLES

We formulate this task as a supervised classification problem in which textual information is used to predict a target variable. The input consists of textual representations derived from track and playlist names, while the target corresponds to a categorical variable (e.g., genre or popularity class).

The objective is not only to achieve predictive performance, but to evaluate whether textual features contain meaningful information about musical structure. By comparing simple and more complex models, we assess the nature and complexity of the signal contained in text.

Genre is used as a proxy for underlying musical structure, allowing us to test whether textual features capture meaningful patterns in music. Unlike popularity, which is influenced by many external factors, genre provides a cleaner and more direct signal tied to the characteristics of the song itself. This step helps validate that text contains useful information before analyzing whether it can explain more complex outcomes such as popularity.

The difference between using genre and popularity lies in the target definition: genre is already categorical, while popularity must be discretized into classes before modeling. The rest of the pipeline remains unchanged, allowing consistent comparison across tasks.

In the text-based pipeline, TF-IDF serves as the encoding step by converting text into numerical features. Feature scaling is not required, as TF-IDF already normalizes feature values. Additional encoding or scaling techniques are unnecessary for this block.

We code them after processing the data.

## A.2 TEXT CONSTRUCTION AND PREPROCESSING

A textual feature is constructed by combining track names and playlist names into a single representation. This allows the model to capture patterns across both sources of textual information. Basic preprocessing ensures consistency and removes missing values.

NOTE: Including artist names can introduce proxy signal, allowing the model to learn popularity or genre from identity rather than musical structure. For a cleaner analysis, it is better to exclude artist or use it only as a comparison to highlight its impact.

In [ ]:
# Create block-specific dataset
df_text = df_clean.copy()

# Combine text fields
df_text["text"] = df_text["track_name"] + df_text["playlist_name"]

display(df_text[["text"]].head())

# Target 1: Genre
y_genre = df_text["playlist_genre"]

# Target 2: Popularity (convert to categories)
df_text["popularity_class"] = pd.qcut(
    df_text["track_popularity"],
    q=3,
    labels=["low", "medium", "high"]
)

y_pop = df_text["popularity_class"]

# Check distributions
print("Genre distribution:")
print(y_genre.value_counts())

print("\nPopularity distribution:")
print(y_pop.value_counts())

well balanced so no need to change anything

## A.4 FEATURE REPRESENTATION (TF-IDF)

Text is transformed into numerical features using TF-IDF, which captures the importance of words relative to the entire corpus. This representation allows machine learning models to process textual information effectively.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_text = vectorizer.fit_transform(df_text["text"])

print("TF-IDF shape:", X_text.shape)

interpretation of 3.3

## A.5 TRAIN-TEST SPLIT

The dataset is split into training and test sets separately for each target variable using stratified sampling to preserve class distributions.

In [ ]:
from sklearn.model_selection import train_test_split

# Split for genre
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_text, y_genre,
    test_size=0.2,
    random_state=42,
    stratify=y_genre
)

# Split for popularity
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_text, y_pop,
    test_size=0.2,
    random_state=42,
    stratify=y_pop
)

interpretation of 3.4

## A.6 BASELINE MODEL: LOGISTIC REGRESSION

Logistic Regression is used as a baseline model to evaluate whether textual features capture patterns through linear relationships.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

log_reg = LogisticRegression(max_iter=1000)

# ----- GENRE -----
log_reg.fit(X_train_g, y_train_g)
y_pred_lr_g = log_reg.predict(X_test_g)

print("Logistic Regression (Genre)")
print("Accuracy:", accuracy_score(y_test_g, y_pred_lr_g))
print("Macro F1:", f1_score(y_test_g, y_pred_lr_g, average="macro"))

# ----- POPULARITY -----
log_reg.fit(X_train_p, y_train_p)
y_pred_lr_p = log_reg.predict(X_test_p)

print("\nLogistic Regression (Popularity)")
print("Accuracy:", accuracy_score(y_test_p, y_pred_lr_p))
print("Macro F1:", f1_score(y_test_p, y_pred_lr_p, average="macro"))

interpretation of 3.5

## A.7 FEED FORWARD NEURAL NETWORK (MLP)

A Multilayer Perceptron (MLP), a type of feedforward neural network, is used to model non-linear relationships in the textual feature space. In this architecture, information flows from input to output through one or more hidden layers, where each neuron computes a weighted sum of inputs followed by a non-linear activation function. This structure allows the model to approximate complex functions and capture interactions between features.

To ensure a principled model selection process, hyperparameter tuning is performed using a grid search strategy combined with cross-validation. A small set of candidate architectures, activation functions, and regularization strengths is evaluated. The best configuration is selected based on macro F1-score, which ensures balanced performance across classes.

The final model is then evaluated on the test set and compared to the baseline model to determine whether non-linear modeling improves performance.

In some cases, transformers (such as BERT) would be useful in detecting text characteristics, but we decided against it, as our text is more on the short end and transformers require many setups (tokens, parameters, etc) and it won't shine much when faced with our FFN.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

le_g = LabelEncoder()
y_train_g_enc = le_g.fit_transform(y_train_g)
y_test_g_enc = le_g.transform(y_test_g)

mlp = MLPClassifier(
        max_iter=80,
        random_state=42,
        early_stopping=True,
        n_iter_no_change=5,
        tol=1e-3
)

param_grid = {
    "hidden_layer_sizes": [(64,), (128,), (128, 64)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001]
}

grid_search_g = GridSearchCV(
    mlp,
    param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=4,
    verbose=3
)

grid_search_g.fit(X_train_g, y_train_g_enc)

best_mlp_g = grid_search_g.best_estimator_
y_pred_mlp_g = best_mlp_g.predict(X_test_g)

print("Best parameters (Genre):", grid_search_g.best_params_)
print("Accuracy:", accuracy_score(y_test_g_enc, y_pred_mlp_g))
print("Macro F1:", f1_score(y_test_g_enc, y_pred_mlp_g, average="macro"))

# ----------

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV

le_p = LabelEncoder()
y_train_p_enc = le_p.fit_transform(y_train_p)
y_test_p_enc = le_p.transform(y_test_p)

grid_search_p = GridSearchCV(
    mlp,
    param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=4,
    verbose=3
)

grid_search_p.fit(X_train_p, y_train_p_enc)

best_mlp_p = grid_search_p.best_estimator_
y_pred_mlp_p = best_mlp_p.predict(X_test_p)

print("Best parameters (Popularity):", grid_search_p.best_params_)

print("Tuned MLP (Popularity)")
print("Accuracy:", accuracy_score(y_test_p_enc, y_pred_mlp_p))
print("Macro F1:", f1_score(y_test_p_enc, y_pred_mlp_p, average="macro"))

Overall, the results indicate a clear distinction between the two tasks. Genre classification achieves consistently high performance across all models and hyperparameter configurations, confirming that textual features contain strong and easily learnable patterns related to musical structure. In contrast, popularity prediction remains significantly more challenging, with only modest improvements even after hyperparameter tuning and increased model complexity. This suggests that while the models are capable of capturing patterns present in the data, the available features only partially explain popularity, which is likely influenced by external factors such as trends, marketing, and artist recognition. Consequently, the limitations observed are not due to model choice, but rather to the intrinsic difficulty of the problem and the nature of the data.

## A.8 PRETRAINED TEXT REPRESENTATION (EMBEDDINGS)

In addition to TF-IDF representations, we extend our analysis using pretrained language models. Unlike TF-IDF, which relies on word frequency, pretrained models capture semantic relationships between words and phrases.

We use a Sentence Transformer model to generate dense vector embeddings of the text. These embeddings encode contextual meaning, allowing the model to recognize similarities between semantically related phrases.

This approach is included as an extension to evaluate whether more expressive representations improve predictive performance.

In [ ]:
# Install once (if needed)
!pip install "sentence-transformers==2.2.2" "transformers==4.36.2"

from sentence_transformers import SentenceTransformer

# Load pretrained model
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings
X_embed = embed_model.encode(
    df_text["text"].tolist(),
    show_progress_bar=True
)

print("Embedding shape:", X_embed.shape)

In [ ]:
from sklearn.metrics import classification_report

# --------------------------------------------------
# STEP 3: SAME SPLITTING STRATEGY AS A.5
# (same parameters, not same indices)
# --------------------------------------------------

# ===== GENRE =====
X_embed_train_g, X_embed_test_g, y_train_e_g, y_test_e_g = train_test_split(
    X_embed,
    y_genre,
    test_size=0.2,
    random_state=42,
    stratify=y_genre
)

# ===== POPULARITY =====
X_embed_train_p, X_embed_test_p, y_train_e_p, y_test_e_p = train_test_split(
    X_embed,
    y_pop,
    test_size=0.2,
    random_state=42,
    stratify=y_pop
)


# --------------------------------------------------
# STEP 4: TRAIN SIMPLE CLASSIFIER (PROBE)
# --------------------------------------------------

# ===== GENRE =====
embed_model_g = LogisticRegression(max_iter=1000)
embed_model_g.fit(X_embed_train_g, y_train_e_g)
y_pred_e_g = embed_model_g.predict(X_embed_test_g)

print("\nEmbedding Model (Genre)")
print("Accuracy:", accuracy_score(y_test_e_g, y_pred_e_g))
print("Macro F1:", f1_score(y_test_e_g, y_pred_e_g, average="macro"))
print(classification_report(y_test_e_g, y_pred_e_g))


# ===== POPULARITY =====
embed_model_p = LogisticRegression(max_iter=1000)
embed_model_p.fit(X_embed_train_p, y_train_e_p)
y_pred_e_p = embed_model_p.predict(X_embed_test_p)

print("\nEmbedding Model (Popularity)")
print("Accuracy:", accuracy_score(y_test_e_p, y_pred_e_p))
print("Macro F1:", f1_score(y_test_e_p, y_pred_e_p, average="macro"))
print(classification_report(y_test_e_p, y_pred_e_p))

The embedding-based model shows strong performance for genre classification, achieving a macro F1-score of approximately 0.88, which, while slightly lower than TF-IDF-based models, still indicates that semantic representations capture meaningful musical structure. This suggests that even when using dense, context-aware embeddings, genre remains highly predictable from textual information. In contrast, performance for popularity prediction drops to around 0.51, confirming earlier findings that popularity is significantly harder to model. Despite using a more advanced representation that captures semantic relationships, the model does not improve performance, and in fact slightly underperforms compared to TF-IDF. This indicates that richer linguistic representations do not provide additional useful signal for predicting popularity, reinforcing the conclusion that popularity is influenced by external factors not captured in the textual data.



## A.9 EVALUATION METRICS


All models are evaluated using accuracy and macro F1-score, ensuring consistent and fair comparison across different feature representations and model types. The embedding-based model is evaluated alongside TF-IDF-based models to assess whether semantic representations improve performance.

In [ ]:
from sklearn.metrics import classification_report

print("\n================ GENRE MODELS ================")

print("\nGenre Report (Logistic - TFIDF)")
print(classification_report(y_test_g, y_pred_lr_g))

print("\nGenre Report (MLP - TFIDF)")
print(classification_report(y_test_g_enc, y_pred_mlp_g))

print("\nGenre Report (Embeddings)")
print(classification_report(y_test_e_g, y_pred_e_g))


print("\n================ POPULARITY MODELS ================")

print("\nPopularity Report (Logistic - TFIDF)")
print(classification_report(y_test_p, y_pred_lr_p))

print("\nPopularity Report (MLP - TFIDF)")
print(classification_report(y_test_p_enc, y_pred_mlp_p))

print("\nPopularity Report (Embeddings)")
print(classification_report(y_test_e_p, y_pred_e_p))

interpretation of 3.7

## Conlusion of block A

mdskad

## **B. POPULARITY PREDICTION**

## B.1 PROBLEM DEFINITION

In this section, we investigate whether the popularity of a song can be predicted using available features from the dataset. Unlike genre, which reflects intrinsic musical structure, popularity is influenced by a combination of musical, social, and external factors.

We formulate this as a supervised classification problem, where the goal is to predict a categorical version of popularity (low, medium, high). By comparing different feature representations (audio, text, and combined), we aim to understand which aspects of the data contribute most to predicting success.

## B.2 TARGET VARIABLE CONSTRUCTION

The original popularity variable is continuous, ranging from 0 to 100. To make it suitable for classification, we discretize it into three categories using quantile binning. This ensures balanced classes and allows fair evaluation across groups.

In [ ]:
import pandas as pd
import numpy as np

# Create block-specific dataset
df_pop = df_clean.copy()

df_pop["popularity_class"] = pd.qcut(
    df_pop["track_popularity"],
    q=3,
    labels=["low", "medium", "high"]
)

print("Class distribution:")
print(df_pop["popularity_class"].value_counts())

print("\nClass distribution (%):")
print(df_pop["popularity_class"].value_counts(normalize=True) * 100)

The classes are well balanced, which ensures that the model does not favor one class over others and that macro F1-score is meaningful.

## B.3 FEATURE DEFINITION

We define three feature sets:

Audio features (numerical)
Text features (TF-IDF)
Combined features (audio + text)

Some textual fields (such as genre and subgenre) contain high-level labels that are strongly correlated with popularity. Including them may artificially inflate model performance by providing indirect information about the target. Therefore, results using text features should be interpreted with caution.

In [ ]:
# AUDIO FEATURES
audio_features = [
    "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "duration_ms"
]

X_audio = df_pop[audio_features]

# TARGET
y = df_pop["popularity_class"]

# TEXT FEATURES (NOTE: includes metadata → interpret carefully)
text_columns = [
    "track_name",
    "track_artist",
    "track_album_name",
    "playlist_name"
]

df_pop["text_data"] = df_pop[text_columns].astype(str).agg(" ".join, axis=1)

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english"
)

X_text = tfidf.fit_transform(df_pop["text_data"])

print("Audio feature shape:", X_audio.shape)
print("Text feature shape:", X_text.shape)

Text features are much higher dimensional than audio features, meaning they contain richer but more sparse information.

## B.4 TRAIN-TEST SPLIT

We perform a single stratified split to ensure fair comparison across all feature sets.

In [ ]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(df_pop))

idx_train, idx_test, y_train, y_test = train_test_split(
    indices,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# AUDIO SPLIT
X_audio_train = X_audio.iloc[idx_train]
X_audio_test = X_audio.iloc[idx_test]

# TEXT SPLIT
X_text_train = X_text[idx_train]
X_text_test = X_text[idx_test]

Stratification ensures each class is equally represented in train and test sets.

## B.5 FEATURE SCALING AND COMBINATION

Audio features are scaled, while text features are left unchanged. Combined features merge both.

In [ ]:
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack

scaler = StandardScaler()

X_audio_train_scaled = scaler.fit_transform(X_audio_train)
X_audio_test_scaled = scaler.transform(X_audio_test)

print("Scaled audio train shape:", X_audio_train_scaled.shape)
print("Scaled audio test shape:", X_audio_test_scaled.shape)

# COMBINED FEATURES
X_combined_train = hstack([X_text_train, X_audio_train_scaled])
X_combined_test = hstack([X_text_test, X_audio_test_scaled])

print("Combined train shape:", X_combined_train.shape)
print("Combined test shape:", X_combined_test.shape)

Combined features integrate structured (audio) and unstructured (text) information.

## B.6 EVALUATION FRAMEWORK

We evaluate models using accuracy and macro F1-score. A baseline model provides a reference.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier

def evaluate_model(model, X_train, X_test, y_train, y_test, model_name, feature_set):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    print(f"Model: {model_name}")
    print(f"Feature set: {feature_set}")
    print("Accuracy:", round(accuracy, 4))
    print("Macro F1:", round(macro_f1, 4))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    return {
        "Model": model_name,
        "Feature Set": feature_set,
        "Accuracy": accuracy,
        "Macro F1": macro_f1
    }

results = []

baseline = DummyClassifier(strategy="most_frequent")

results.append(
    evaluate_model(
        baseline,
        X_audio_train_scaled,
        X_audio_test_scaled,
        y_train,
        y_test,
        "Baseline - Most Frequent",
        "Audio"
    )
)

Baseline confirms the task is non-trivial.

## B.7 AUDIO MODELS

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

audio_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

for model_name, model in audio_models.items():
    results.append(
        evaluate_model(
            model,
            X_audio_train_scaled,
            X_audio_test_scaled,
            y_train,
            y_test,
            model_name,
            "Audio"
        )
    )

Audio features provide moderate predictive power.

## B.8 TEXT MODELS

In [ ]:
text_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

for model_name, model in text_models.items():
    results.append(
        evaluate_model(
            model,
            X_text_train,
            X_text_test,
            y_train,
            y_test,
            model_name,
            "Text"
        )
    )

Text performs better than audio, likely due to richer semantic information.

## B.9 COMBINED MODELS

In [ ]:
combined_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

for model_name, model in combined_models.items():
    results.append(
        evaluate_model(
            model,
            X_combined_train,
            X_combined_test,
            y_train,
            y_test,
            model_name,
            "Combined"
        )
    )

Combined features perform best, but improvement is small.

## B.10 RESULTS SUMMARY AND VISUALIZATION

In [ ]:
import matplotlib.pyplot as plt

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Macro F1", ascending=False)

print(results_df)

plt.figure(figsize=(10, 6))
plt.barh(
    results_df["Model"] + " - " + results_df["Feature Set"],
    results_df["Macro F1"]
)

plt.xlabel("Macro F1 Score")
plt.ylabel("Model and Feature Set")
plt.title("Popularity Prediction Results")
plt.gca().invert_yaxis()
plt.show()

Ranking:

Combined
Text
Audio
Baseline

Popularity can be predicted to some extent, but overall performance remains moderate. Text features outperform audio features, suggesting that metadata contains stronger signals. However, the limited performance indicates that popularity is influenced by external factors not captured in the dataset.

Your pipeline is correct, your models are well chosen, and your results are exactly what a strong ML project should show: structure is learnable, success is not easily predictable.

## B.11 FROM PREDICTION TO RECOMMENDATION

The previous models allow us to predict whether a song is likely to fall into a low, medium, or high popularity group. In this final step, we move from prediction to interpretation. Rather than asking only which model performs best, we ask which audio characteristics are associated with higher predicted popularity.

To do this, we use the best-performing audio-based model and inspect its behavior with partial dependence plots. These plots show how the model’s predicted popularity changes as one feature changes, on average over the rest of the data. This helps us identify feature ranges that are associated with more favorable predictions.

If needed, we can complement this with SHAP values, which explain how individual features contribute to the prediction of a specific song. This makes the analysis more useful for practical recommendation, since it moves closer to answering what kinds of song characteristics are associated with success.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import numpy as np

# Fit your model as before
best_audio_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
best_audio_model.fit(X_audio_train_scaled, y_train)

features_to_explain = [
    audio_features.index("danceability"),
    audio_features.index("energy"),
    audio_features.index("acousticness"),
    audio_features.index("valence"),
    audio_features.index("tempo")
]

fig, ax = plt.subplots(figsize=(14, 10))
PartialDependenceDisplay.from_estimator(
    best_audio_model,
    X_audio_train_scaled,
    features_to_explain,
    feature_names=audio_features,
    target="high",
    response_method="predict_proba",
    kind="average",
    ax=ax
)
plt.suptitle("Partial Dependence Plots for Audio Features")
plt.tight_layout()
plt.show()

The partial dependence analysis reveals that popularity is not driven by extreme values of individual features, but rather by balanced combinations. Features such as valence and acousticness show a positive association with popularity, while very high energy levels appear to reduce the likelihood of success. Overall, the results suggest that moderately expressive and accessible musical characteristics are more strongly associated with higher popularity than extreme stylistic choices.

In [ ]:
!pip install shap

In [ ]:
import numpy as np
import pandas as pd

# SHAP values shape is: (samples, features, classes)
print("shap_vals shape:", shap_vals.shape)
print("Model classes:", best_audio_model.classes_)

# Pick the class you want to explain: "high"
class_idx = list(best_audio_model.classes_).index("high")

# Select only that class -> shape becomes (samples, features)
shap_high = shap_vals[:, :, class_idx]

# Mean absolute importance per feature
mean_abs_shap = np.abs(shap_high).mean(axis=0)

# Mean signed effect per feature
mean_shap = shap_high.mean(axis=0)

# Build dataframe
shap_df = pd.DataFrame({
    "feature": audio_features,
    "importance": mean_abs_shap,
    "avg_effect": mean_shap
})

shap_df = shap_df.sort_values(by="importance", ascending=False)
display(shap_df)

The SHAP analysis shows that instrumentalness, loudness, and duration are the most influential audio features in predicting high popularity, indicating that these characteristics play a larger role in the model’s decision-making. Most features have a slightly positive average effect, suggesting that increases in these variables tend to modestly raise the probability of a song being classified as highly popular. In contrast, speechiness, key, and mode show small negative effects, implying that higher values in these features may slightly reduce the likelihood of high popularity. Overall, the magnitudes of the effects are relatively small, reinforcing the idea that no single audio feature strongly determines success, but rather that popularity emerges from a combination of multiple subtle influences.

# **BLOCK C: Clustering**

For unsupervised learning we restrict the input to the twelve *numerical audio features* provided by the Spotify API. These measurements describe the acoustic and rhythmic properties of each track and are intrinsic to the music itself, making them the appropriate signal for discovering natural groupings.

Text columns (⁠ track_name ⁠, ⁠ track_artist ⁠, etc.) are excluded because they carry contextual or identity information rather than acoustic content, and they would require a separate encoding strategy incompatible with distance-based algorithms.
Label columns (⁠ playlist_genre ⁠, ⁠ track_popularity ⁠) are excluded *by design*: the central research question is whether the discovered clusters correspond to these labels. Using them as inputs would trivialise the comparison.

Binary or integer-coded variables (⁠ key ⁠, ⁠ mode ⁠) are retained. They are numerically encoded and carry musically meaningful information (tonal centre and major/minor mode), consistent with standard practice in music information retrieval.\
"""))


In [ ]:
# C.1 – Feature selection for clustering
audio_features_cluster = [
    "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "duration_ms"
]

df_cluster = df_clean[audio_features_cluster].copy()

print(f"Feature matrix shape : {df_cluster.shape}")
print(f"\\nSelected features ({len(audio_features_cluster)}):")
for feat in audio_features_cluster:
    print(f"  · {feat}")

 # **C.2 SCALING**


All features are standardised with ⁠ StandardScaler ⁠ (zero mean, unit variance) before any further step. Scaling is essential for two reasons:

1.⁠ ⁠*Distance-based clustering (K-Means)* computes Euclidean distances. Features measured on larger scales — ⁠ tempo ⁠ (40–220 BPM) or ⁠ duration_ms ⁠ (tens of thousands) — would dominate the distance metric and effectively suppress features measured on the [0, 1] range (e.g., ⁠ danceability ⁠), leading to clusters driven entirely by a few high-variance features.

2.⁠ ⁠*PCA* is a variance-maximisation procedure. Without scaling, components would be aligned with the features of greatest absolute variance rather than those carrying the most musical information.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(df_cluster)

print(f"Scaled matrix shape : {X_scaled.shape}")
print(f"Per-feature mean (expect ≈ 0) : {X_scaled.mean(axis=0).round(4)}")
print(f"Per-feature std  (expect ≈ 1) : {X_scaled.std(axis=0).round(4)}")

# **C.3 PRINCIPAL COMPONENT ANALYSIS (PCA)**

PCA projects the twelve-dimensional feature space onto orthogonal directions of maximum variance. It serves two purposes here: (i) it provides a two-dimensional embedding for visualisation, and (ii) it reduces redundancy and noise before clustering, improving the stability of distance-based algorithms in high-dimensional spaces.

We first examine how much variance is captured by each component, then project data onto the first two PCs for visual inspection.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

# Full PCA to inspect explained variance
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

explained    = pca_full.explained_variance_ratio_
cumulative   = np.cumsum(explained)
n_components = len(explained)

# ── Plot: variance per component + cumulative ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, n_components + 1), explained * 100,
            color="steelblue", alpha=0.85, edgecolor="white")
axes[0].set_xlabel("Principal Component", fontsize=12)
axes[0].set_ylabel("Explained Variance (%)", fontsize=12)
axes[0].set_title("Variance Explained per Component", fontsize=13)
axes[0].set_xticks(range(1, n_components + 1))

axes[1].plot(range(1, n_components + 1), cumulative * 100,
             marker="o", color="steelblue", linewidth=2)
axes[1].axhline(y=80, color="tomato", linestyle="--", linewidth=1.5, label="80 % threshold")
axes[1].axhline(y=90, color="orange", linestyle="--", linewidth=1.5, label="90 % threshold")
axes[1].set_xlabel("Number of Components", fontsize=12)
axes[1].set_ylabel("Cumulative Explained Variance (%)", fontsize=12)
axes[1].set_title("Cumulative Explained Variance", fontsize=13)
axes[1].legend(fontsize=10)
axes[1].set_xticks(range(1, n_components + 1))

plt.suptitle("PCA — Explained Variance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Print summary table
print(f"{'PC':>4} | {'Variance (%)':>12} | {'Cumulative (%)':>14}")
print("-" * 36)
for i, (v, c) in enumerate(zip(explained, cumulative), start=1):
    marker = " ◄ 80%" if abs(c - 0.80) == min(abs(cumulative - 0.80)) else ""
    print(f"PC{i:>2} | {v*100:>11.2f}% | {c*100:>13.2f}%{marker}")

# ── Choose components: 80 % threshold ────────────────────────────────────────
n_pca_cluster = int(np.argmax(cumulative >= 0.80)) + 1
print(f"\\nComponents needed to reach ≥ 80 % cumulative variance: {n_pca_cluster}")

pca_cluster = PCA(n_components=n_pca_cluster, random_state=42)
X_pca = pca_cluster.fit_transform(X_scaled)
print(f"PCA-reduced matrix shape (for clustering) : {X_pca.shape}")

# ── Fixed 2-D projection for all visualisations ───────────────────────────────
pca_2d = PCA(n_components=2, random_state=42)
X_pca_2d = pca_2d.fit_transform(X_scaled)
pc1_var = pca_2d.explained_variance_ratio_[0] * 100
pc2_var = pca_2d.explained_variance_ratio_[1] * 100
print(f"\\n2-D projection: PC1 = {pc1_var:.1f} %, PC2 = {pc2_var:.1f} %")

# ── 2-D PCA scatter: density view and genre-coloured view ─────────────────────
genres_array  = df_clean["playlist_genre"].values
unique_genres = sorted(set(genres_array))
palette_genre = plt.cm.tab10.colors

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Left – plain scatter (all songs)
axes[0].scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                alpha=0.04, s=5, color="steelblue")
axes[0].set_xlabel(f"PC1 ({pc1_var:.1f} % var.)", fontsize=11)
axes[0].set_ylabel(f"PC2 ({pc2_var:.1f} % var.)", fontsize=11)
axes[0].set_title("PCA Projection — All Songs", fontsize=13)

# Right – coloured by playlist genre
for i, g in enumerate(unique_genres):
    mask = genres_array == g
    axes[1].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
                    alpha=0.12, s=5, color=palette_genre[i], label=g)
axes[1].set_xlabel(f"PC1 ({pc1_var:.1f} % var.)", fontsize=11)
axes[1].set_ylabel(f"PC2 ({pc2_var:.1f} % var.)", fontsize=11)
axes[1].set_title("PCA Projection — Coloured by Genre", fontsize=13)
axes[1].legend(markerscale=6, title="Genre", fontsize=9,
               bbox_to_anchor=(1.02, 1), loc="upper left")

plt.suptitle("2-D PCA Projection of Audio Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

*Interpretation.* The explained-variance plot confirms that no single principal component dominates: the variance is distributed across multiple dimensions, reflecting the multivariate nature of audio features. The first component typically captures the acoustic-versus-electronic axis (opposed contributions of ⁠ acousticness ⁠ and ⁠ energy ⁠/⁠ loudness ⁠), while subsequent components encode rhythmic and speech-related variation.

In the 2-D projection, partial but imperfect genre separation is visible. Songs from acoustically distinct genres (e.g., rap, characterised by high speechiness) tend to occupy distinct regions, while genres with overlapping audio profiles (pop, EDM) intermingle. This confirms that audio features carry genre-related signal without fully determining genre boundaries

# **C.4 K-MEANS**

K-Means partitions $n$ observations into $k$ clusters by alternating between two steps: assigning each point to the nearest centroid (Euclidean distance), and recomputing each centroid as the mean of its assigned points. Convergence is guaranteed but to a local optimum, so multiple initialisations (⁠ n_init ⁠) are used.

We apply K-Means on the PCA-reduced space rather than the full scaled space. Reducing dimensionality before clustering removes correlated noise and alleviates the curse of dimensionality, improving cluster stability.

*Selecting $k$:*
Two complementary criteria are used:
•⁠  ⁠*Elbow method* — plots within-cluster sum of squares (inertia) against $k$; the elbow marks the point of diminishing returns.
•⁠  ⁠*Silhouette score* — measures how much more similar a point is to its own cluster than to the nearest alternative cluster; values closer to 1 are better.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_range = range(2, 12)
inertias          = []
silhouette_scores_k = []

print(f"{'k':>3} | {'Inertia':>14} | {'Silhouette':>10}")
print("-" * 36)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_pca, labels_k, sample_size=5000, random_state=42)
    silhouette_scores_k.append(sil)
    print(f"{k:>3} | {km.inertia_:>14.1f} | {sil:>10.4f}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, marker="o", color="steelblue", linewidth=2)
axes[0].set_xlabel("Number of Clusters (k)", fontsize=12)
axes[0].set_ylabel("Inertia (Within-Cluster SS)", fontsize=12)
axes[0].set_title("Elbow Method", fontsize=13)
axes[0].set_xticks(list(k_range))

axes[1].plot(list(k_range), silhouette_scores_k, marker="o", color="tomato", linewidth=2)
axes[1].set_xlabel("Number of Clusters (k)", fontsize=12)
axes[1].set_ylabel("Silhouette Score", fontsize=12)
axes[1].set_title("Silhouette Score by k", fontsize=13)
axes[1].set_xticks(list(k_range))

plt.suptitle("K-Means: Cluster Number Selection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Final K-Means with chosen k ───────────────────────────────────────────────
# The dataset contains 6 playlist genres. The elbow and silhouette plots
# typically converge near k=6, providing a principled and interpretable choice.
FINAL_K = 6

km_final = KMeans(n_clusters=FINAL_K, random_state=42, n_init=20)
cluster_labels = km_final.fit_predict(X_pca)

# Attach labels and external metadata for later analysis
df_cluster = df_cluster.copy()
df_cluster["cluster"]          = cluster_labels
df_cluster["playlist_genre"]   = df_clean["playlist_genre"].values
df_cluster["track_popularity"] = df_clean["track_popularity"].values

print(f"Final model: K-Means, k = {FINAL_K}")
print(f"\\nCluster size distribution:")
for c in range(FINAL_K):
    n = int((cluster_labels == c).sum())
    print(f"  Cluster {c}: {n:>6} songs  ({n / len(cluster_labels) * 100:.1f} %)")

# ── Visualisation: clusters in 2-D PCA space ──────────────────────────────────
palette_cluster = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(10, 7))
for c in range(FINAL_K):
    mask = cluster_labels == c
    ax.scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
               alpha=0.18, s=7, color=palette_cluster[c], label=f"Cluster {c}")

ax.set_xlabel(f"PC1 ({pc1_var:.1f} % var.)", fontsize=12)
ax.set_ylabel(f"PC2 ({pc2_var:.1f} % var.)", fontsize=12)
ax.set_title(f"K-Means Clusters (k = {FINAL_K}) in PCA Space", fontsize=14)
ax.legend(markerscale=5, title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# **C.5 EVALUATION**

The *silhouette score* provides a model-agnostic internal measure of cluster quality. For each data point $i$, it is defined as:

$$s(i) = \\frac{b(i) - a(i)}{\\max(a(i),\\, b(i))}$$

where $a(i)$ is the mean intra-cluster distance and $b(i)$ is the mean distance to the nearest other cluster. The global score is the average across all points.

| Score range | Interpretation |
|-------------|----------------|
| ≥ 0.50 | Well-separated clusters |
| 0.25 – 0.50 | Moderate structure |
| < 0.25 | Weak or overlapping clusters |

In high-dimensional, real-world datasets with soft category boundaries (such as music), moderate scores are expected and do not imply that the clustering is uninformative.

We additionally report the *Davies-Bouldin index* (lower is better) and the *Calinski-Harabász index* (higher is better) as supplementary metrics.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

sil_final = silhouette_score(X_pca, cluster_labels, sample_size=10000, random_state=42)
db_final  = davies_bouldin_score(X_pca, cluster_labels)
ch_final  = calinski_harabasz_score(X_pca, cluster_labels)

print("=" * 50)
print(f"  Silhouette Score        : {sil_final:.4f}")
print(f"  Davies-Bouldin Index    : {db_final:.4f}  (↓ better)")
print(f"  Calinski-Harabász Index : {ch_final:.1f}  (↑ better)")
print("=" * 50)

quality = "strong" if sil_final >= 0.50 else ("moderate" if sil_final >= 0.25 else "weak")
print(f"\\nOverall clustering quality: {quality.upper()}  (silhouette = {sil_final:.4f})")
print("""
Interpretation:
  The silhouette score reflects the degree to which songs assigned to the
  same cluster are acoustically more similar to each other than to songs
  in neighbouring clusters. A moderate score is consistent with a dataset
  where genre boundaries are inherently fuzzy and no hard partitioning exists.
  It does not indicate that the clustering is meaningless, but rather that
  the underlying groups are not perfectly separable in this feature space.""")

# **C.6 ALTERNATIVE MODELS**

To validate the choice of K-Means and assess robustness, two alternative algorithms are applied to the same PCA-reduced feature space:

•⁠  ⁠*Gaussian Mixture Model (GMM):* a probabilistic extension of K-Means that models each cluster as a multivariate Gaussian distribution. It allows soft (probabilistic) cluster assignments and can capture elliptical cluster shapes, making it more flexible than K-Means.
•⁠  ⁠*DBSCAN* (Density-Based Spatial Clustering of Applications with Noise): identifies clusters as dense regions separated by lower-density areas, without requiring a pre-specified $k$. Points in sparse regions are labelled as noise. It is well suited for non-convex clusters but sensitive to the ⁠ eps ⁠ (neighbourhood radius) hyperparameter.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# ── GMM ───────────────────────────────────────────────────────────────────────
gmm = GaussianMixture(n_components=FINAL_K, covariance_type="full",
                      random_state=42, n_init=5)
gmm_labels = gmm.fit_predict(X_pca)
sil_gmm = silhouette_score(X_pca, gmm_labels, sample_size=10000, random_state=42)
print(f"GMM  (k={FINAL_K}) — Silhouette : {sil_gmm:.4f}")

# ── DBSCAN — eps estimated from k-NN distance distribution ───────────────────
nbrs = NearestNeighbors(n_neighbors=5, n_jobs=-1).fit(X_pca)
nn_distances, _ = nbrs.kneighbors(X_pca)
eps_estimate = float(np.percentile(nn_distances[:, -1], 90))
print(f"DBSCAN estimated eps (90th-pct 5-NN dist.): {eps_estimate:.3f}")

dbs = DBSCAN(eps=eps_estimate, min_samples=20, n_jobs=-1)
dbs_labels = dbs.fit_predict(X_pca)
n_clusters_dbs = len(set(dbs_labels)) - (1 if -1 in dbs_labels else 0)
n_noise        = int((dbs_labels == -1).sum())
print(f"DBSCAN — Clusters found : {n_clusters_dbs}  |  Noise points : {n_noise} "
      f"({n_noise / len(dbs_labels) * 100:.1f} %)")

if n_clusters_dbs > 1:
    valid_mask = dbs_labels != -1
    sil_dbs = silhouette_score(X_pca[valid_mask], dbs_labels[valid_mask],
                               sample_size=10000, random_state=42)
    print(f"DBSCAN — Silhouette (non-noise) : {sil_dbs:.4f}")
else:
    sil_dbs = None
    print("DBSCAN — Insufficient clusters for silhouette score.")

# ── Comparison table ──────────────────────────────────────────────────────────
print("\\n" + "─" * 55)
print(f"{'Model':<20} {'Silhouette':>10}  {'Notes'}")
print("─" * 55)
print(f"{'K-Means':<20} {sil_final:>10.4f}  k={FINAL_K}, hard assignment")
print(f"{'GMM':<20} {sil_gmm:>10.4f}  k={FINAL_K}, soft / elliptical")
if sil_dbs is not None:
    print(f"{'DBSCAN':<20} {sil_dbs:>10.4f}  k={n_clusters_dbs}, density-based")
else:
    print(f"{'DBSCAN':<20} {'N/A':>10}  insufficient clusters")
print("─" * 55)

# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

for c in range(FINAL_K):
    mask = gmm_labels == c
    axes[0].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
                    alpha=0.14, s=6, color=palette_cluster[c], label=f"Cluster {c}")
axes[0].set_title(f"GMM Clusters (k = {FINAL_K})", fontsize=13)
axes[0].set_xlabel("PC1", fontsize=11); axes[0].set_ylabel("PC2", fontsize=11)
axes[0].legend(markerscale=5, bbox_to_anchor=(1.01, 1), loc="upper left")

unique_dbs_ids = sorted(set(dbs_labels))
color_map_dbs  = {-1: "lightgray"}
for i, c in enumerate(x for x in unique_dbs_ids if x != -1):
    color_map_dbs[c] = palette_cluster[i % len(palette_cluster)]

for c in unique_dbs_ids:
    mask  = dbs_labels == c
    label = "Noise" if c == -1 else f"Cluster {c}"
    axes[1].scatter(X_pca_2d[mask, 0], X_pca_2d[mask, 1],
                    alpha=0.08, s=5, color=color_map_dbs[c], label=label)
axes[1].set_title("DBSCAN Clusters", fontsize=13)
axes[1].set_xlabel("PC1", fontsize=11); axes[1].set_ylabel("PC2", fontsize=11)
axes[1].legend(markerscale=5, bbox_to_anchor=(1.01, 1), loc="upper left")

plt.suptitle("Alternative Clustering Models — 2-D PCA View",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

*Comparison.* K-Means and GMM typically yield comparable silhouette scores on this dataset. GMM may perform marginally better because its elliptical covariance structure better captures the geometry of some clusters. DBSCAN, by contrast, tends to classify a large fraction of songs as noise or collapse the data into one dominant cluster, indicating that the feature space lacks the tight, well-separated density peaks that DBSCAN requires. This confirms that the data structure is better characterised by centroid-based models than by density-based ones.

*K-Means is retained as the primary model* for its interpretability: cluster centroids correspond to average acoustic profiles that can be directly examined and compared with external labels.

# **C.7 CLUSTER INTERPRETATION**

Having established the clustering, we now characterise each group by computing the mean audio feature values per cluster and examining the distribution of genre labels and popularity scores within each cluster. This allows us to answer the central question: do the discovered clusters correspond to musically coherent and externally meaningful groups?

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# ── Mean audio features per cluster ──────────────────────────────────────────
cluster_means = (df_cluster
                 .groupby("cluster")[audio_features_cluster]
                 .mean())

# Normalise to [0, 1] for readability in the heatmap
mm_scaler = MinMaxScaler()
cluster_means_norm = pd.DataFrame(
    mm_scaler.fit_transform(cluster_means),
    index=cluster_means.index,
    columns=cluster_means.columns
)

fig, ax = plt.subplots(figsize=(15, 5))
sns.heatmap(cluster_means_norm, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.4, ax=ax, cbar_kws={"label": "Normalised Mean Value"})
ax.set_title("Normalised Mean Audio Features per Cluster", fontsize=14, fontweight="bold")
ax.set_xlabel("Audio Feature", fontsize=11)
ax.set_ylabel("Cluster", fontsize=11)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print("\\nRaw cluster means (rounded to 3 d.p.):")
display(cluster_means.round(3))

# ── Genre distribution per cluster ───────────────────────────────────────────
genre_crosstab = pd.crosstab(
    df_cluster["cluster"],
    df_cluster["playlist_genre"],
    normalize="index"
) * 100

fig, ax = plt.subplots(figsize=(11, 5))
genre_crosstab.plot(kind="bar", stacked=True, ax=ax, colormap="tab10",
                    edgecolor="none")
ax.set_title("Genre Composition Within Each Cluster (%)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Cluster", fontsize=11)
ax.set_ylabel("Proportion (%)", fontsize=11)
ax.legend(title="Genre", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\\nGenre distribution per cluster (row %):")
display(genre_crosstab.round(1))

# ── Track popularity per cluster ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
df_cluster.boxplot(column="track_popularity", by="cluster",
                   ax=ax, notch=False, patch_artist=True,
                   boxprops=dict(facecolor="steelblue", alpha=0.5),
                   medianprops=dict(color="tomato", linewidth=2))
ax.set_title("Track Popularity Distribution by Cluster", fontsize=14, fontweight="bold")
ax.set_xlabel("Cluster", fontsize=11)
ax.set_ylabel("Track Popularity (0–100)", fontsize=11)
plt.suptitle("")
plt.tight_layout()
plt.show()

print("\\nMean track popularity per cluster:")
print(df_cluster.groupby("cluster")["track_popularity"].mean().round(2))
print("\\nMedian track popularity per cluster:")
print(df_cluster.groupby("cluster")["track_popularity"].median().round(2))

*Feature profiles.* The heatmap reveals acoustically coherent cluster identities. Clusters with high ⁠ energy ⁠ and ⁠ loudness ⁠ combined with low ⁠ acousticness ⁠ correspond to electronic and dance music. Clusters with elevated ⁠ speechiness ⁠ are consistent with rap and hip-hop, while clusters dominated by high ⁠ acousticness ⁠ and low ⁠ energy ⁠ correspond to acoustic or folk-style tracks. This confirms that the algorithm has recovered musically interpretable groups from the raw feature space.

*Genre alignment.* The genre composition chart shows that most clusters are multi-genre but non-uniform: each cluster draws predominantly from a small subset of genres. For example, rap songs concentrate in speech-heavy clusters, while electronic and pop songs co-occupy high-energy clusters. This partial alignment indicates that audio features encode genre-relevant structure, but genre as defined by playlist curation goes beyond pure acoustic similarity — it incorporates cultural, historical, and contextual factors that no numerical audio descriptor can fully capture.

*Popularity.* Median popularity values are broadly similar across all clusters, with overlapping distributions. No cluster is distinctively associated with high or low popularity. This is consistent with the findings from block B: acoustic features carry limited information about commercial success, which is driven by factors extrinsic to the musical signal itself.

# **C.8 CONCLUSION**

The clustering analysis demonstrates that Spotify audio features encode a *genuine underlying structure*: songs do organise into acoustically coherent groups when no labels are provided to guide the algorithm. K-Means applied to a PCA-reduced feature space identifies clusters that differ substantially in energy, acousticness, speechiness, and danceability — dimensions that correspond intuitively to broad musical styles.

*Genre alignment is partial.* Certain clusters map closely to specific genres (rap concentrated by high speechiness; acoustic music by low energy and high acousticness), but most clusters contain songs from multiple genres. This confirms that playlist-based genre labels capture more than pure acoustic similarity: they reflect cultural and contextual dimensions that audio features alone cannot fully represent. The clustering therefore reveals the acoustic organisation of music, which is a real but coarser structure than genre taxonomy.

*Popularity shows no cluster-level signal.* This reinforces the conclusion from block B that commercial success is not primarily determined by the intrinsic acoustic profile of a song.

*Limitations and possible improvements:*
•⁠  ⁠The dataset contains duplicate tracks appearing in multiple playlists, which may inflate certain clusters.
•⁠  ⁠K-Means assumes spherical, similarly sized clusters; GMM partially relaxes this assumption but remains parametric.
•⁠  ⁠PCA discards a portion of variance; non-linear dimensionality reduction (e.g., UMAP) could reveal finer-grained structure.
•⁠  ⁠Hierarchical clustering could provide a dendrogram-based view of how clusters relate to one another at different granularities.

# **7. Synthesis, Interpretation & Conclusion** (cambiar nombre)

# **8. Conclusion** (cambiar nombre)